# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as defined by the [FAIR\^2 data package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata. mlcroissant's metadata object exposes fields as attributes.
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("Identifier:", getattr(metadata, 'identifier', None))
print("Version:", getattr(metadata, 'version', None))
print("Keywords:", getattr(metadata, 'keywords', None))
print("License:", getattr(metadata, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their `@id` references. This is crucial for extracting data by referencing the appropriate entity IDs.

We enumerate all record sets (tables/files), their `@id`, and example fields/columns.

In [ ]:
# List all record sets in the dataset by @id
record_sets = dataset.metadata.record_sets
if record_sets:
    for record_set in record_sets:
        print(f"Record set: {record_set.name}")
        print(f"  @id: {record_set.id}")
        print(f"  Description: {getattr(record_set, 'description', '')}")
        # Print up to 10 fields/columns by @id
        if hasattr(record_set, 'fields') and record_set.fields:
            print("  Fields/columns (by @id):")
            for field in record_set.fields[:10]:
                collist = getattr(field, 'columns', [])
                colids = [c.id for c in collist] if collist else []
                print(f"    - {field.name} (@id: {field.id}, columns: {colids})")
        print()
else:
    print("No record sets found in this Croissant metadata. Please check the data structure.")

## 3. Data Extraction
Extract data from a record set into a DataFrame for analysis.

Select the primary table of interest by its `@id`. For this dataset, the main data table often contains proteomics measurements for the extracellular vesicle samples.

**Remember:** Replace `<record_set_id>` below with the appropriate `@id` from the output above.

In [ ]:
# Example: List all record set IDs and pick one for analysis
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets] if record_sets else []
print('Available record_set @id values:', record_set_ids)

# Choose the main data record set @id (update manually or programmatically as needed)
# Example: Suppose the main record set is '@id': 'https://api.app.sen.science/frontiers/7154140/2f0414ee-8dd2-439f-af70-acd989d6a7e1'
# Replace with the correct @id from the above list
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print(f"Columns in {main_record_set_id}:", df.columns.tolist())
    display(df.head())
else:
    print("No record set selected for extraction.")

## 4. Exploratory Data Analysis (EDA)
Now, let's conduct basic EDA:
- Filter records with large or small values in some numeric field (`@id` specified below)
- Normalize a numeric field
- Group by key identifiers

**Edit the `numeric_field_id` and `group_field_id` with real field names or `@id`s from your data.**

In [ ]:
# Choose a numeric field for filtering and normalization (update as needed)
if 'df' in locals() and not df.empty:
    print("Sample columns:", df.columns.tolist())
    # Example: Assume a field like 'MW_kDa' or 'Abundance' exists
    # Replace below with your field's name or column (by @id)
    numeric_field_id = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else None
    if numeric_field_id:
        threshold = df[numeric_field_id].quantile(0.9)  # Use the top 10% as an example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f'{numeric_field_id}_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

        # Choose a group field if any are categorical/object type columns
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field detected in the data.")
else:
    print("DataFrame is empty or not loaded.")

## 5. Visualization
Visualize distributions or relationships between fields using Matplotlib or Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping by a categorical field exists, plot a boxplot
    if 'group_field_id' in locals() and group_field_id in filtered_df:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Filtered data not available for visualization.")

## 6. Conclusion
In this notebook, we've loaded, explored, and visualized the FAIR^2 Croissant dataset representing mass spectrometry measurements from human extracellular vesicles. Fields and columns were referenced by their explicit `@id`. You can continue to analyze or preprocess specific fields for downstream tasks such as machine learning, biomarker search, or domain-specific studies.

Refer to [mlcroissant documentation](https://mlcommons.github.io/croissant/python/autoapi/mlcroissant/) for additional dataset manipulation and metadata queries.